### Sliding window test
- window_size = 60s
- step_size = 30s
- w, w/o discard: 윈도우 내에서 stress 비율이 10미만일때 discard

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError

DATA_ROOT = r"/content/drive/MyDrive/Colab Notebooks/datasets/Wearable-device-dataset/Wearable_Dataset"
STRESS_ROOT = os.path.join(DATA_ROOT, "STRESS")

WINDOW_SEC = 60
STEP_SEC = 30

# discard 방식 기준
STRESS_RATIO_THR = 0.50
REST_RATIO_THR = 0.10


def read_bvp_meta(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    start_time = pd.to_datetime(lines[0])
    fs = float(lines[1])
    n_samples = len(lines) - 2
    duration_sec = n_samples / fs
    end_time = start_time + pd.to_timedelta(duration_sec, unit="s")

    return start_time, end_time, fs, n_samples, duration_sec


def read_tags(path):
    if os.path.getsize(path) == 0:
        return []

    try:
        tags = pd.read_csv(path, header=None)
    except EmptyDataError:
        return []

    if tags.empty:
        return []

    tags[0] = pd.to_datetime(tags[0])
    return tags[0].tolist()


def sec_to_minsec(sec):
    m = int(sec // 60)
    s = int(round(sec % 60))
    return f"{m}:{s:02d}"


def overlap_sec(a_start, a_end, b_start, b_end):
    start = max(a_start, b_start)
    end = min(a_end, b_end)
    sec = (end - start).total_seconds()
    return max(0.0, sec)


def build_v2_intervals(tags):
    """
    Baseline 제외.
    v2 protocol:
    TMCT             tag[1]~tag[2] stress
    First Rest       tag[2]~tag[3] rest
    Real Opinion     tag[3]~tag[4] stress
    Opposite Opinion tag[5]~tag[6] stress
    Second Rest      tag[6]~tag[7] rest
    Subtract Test    tag[7]~tag[8] stress
    """
    if len(tags) < 9:
        raise ValueError(f"v2 requires at least 9 tags, got {len(tags)}")

    return pd.DataFrame([
        {"phase": "TMCT",             "start": tags[1], "end": tags[2], "label": 1},
        {"phase": "First Rest",       "start": tags[2], "end": tags[3], "label": 0},
        {"phase": "Real Opinion",     "start": tags[3], "end": tags[4], "label": 1},
        {"phase": "Opposite Opinion", "start": tags[5], "end": tags[6], "label": 1},
        {"phase": "Second Rest",      "start": tags[6], "end": tags[7], "label": 0},
        {"phase": "Subtract Test",    "start": tags[7], "end": tags[8], "label": 1},
    ])


def make_timeline_windows(intervals_df, window_sec=60, step_sec=30):
    rows_no_discard = []
    rows_with_discard = []

    timeline_start = intervals_df["start"].min()
    timeline_end = intervals_df["end"].max()

    win_delta = pd.to_timedelta(window_sec, unit="s")
    step_delta = pd.to_timedelta(step_sec, unit="s")

    t0 = timeline_start
    window_idx = 0

    while t0 + win_delta <= timeline_end:
        t1 = t0 + win_delta
        window_idx += 1

        stress_overlap = 0.0
        rest_overlap = 0.0
        phase_overlaps = {}

        for _, r in intervals_df.iterrows():
            ov = overlap_sec(t0, t1, r["start"], r["end"])

            if ov <= 0:
                continue

            phase_overlaps[r["phase"]] = ov

            if int(r["label"]) == 1:
                stress_overlap += ov
            else:
                rest_overlap += ov

        covered_sec = stress_overlap + rest_overlap
        stress_ratio = stress_overlap / window_sec
        rest_ratio = rest_overlap / window_sec
        covered_ratio = covered_sec / window_sec

        # 방식 1: discard 미포함
        # stress가 rest보다 많으면 stress, 아니면 rest
        if stress_overlap >= rest_overlap:
            label_no_discard = 1
            status_no_discard = "stress"
        else:
            label_no_discard = 0
            status_no_discard = "normal"

        rows_no_discard.append({
            "window_index": window_idx,
            "window_start": t0,
            "window_end": t1,
            "label": label_no_discard,
            "status_name": status_no_discard,
            "stress_overlap_sec": stress_overlap,
            "rest_overlap_sec": rest_overlap,
            "stress_ratio": stress_ratio,
            "rest_ratio": rest_ratio,
            "covered_ratio": covered_ratio,
            "dominant_phase": max(phase_overlaps, key=phase_overlaps.get) if phase_overlaps else "none",
            "phase_overlaps": str(phase_overlaps),
        })

        # 방식 2: discard 포함
        if stress_ratio >= STRESS_RATIO_THR:
            label = 1
            status = "stress"
            keep = 1
            reason = "stress_ratio>=0.5"
        elif stress_ratio <= REST_RATIO_THR and rest_ratio > 0:
            label = 0
            status = "normal"
            keep = 1
            reason = "stress_ratio<=0.1"
        else:
            label = -1
            status = "discard"
            keep = 0
            reason = "ambiguous"

        rows_with_discard.append({
            "window_index": window_idx,
            "window_start": t0,
            "window_end": t1,
            "label": label,
            "status_name": status,
            "keep": keep,
            "discard_reason": reason,
            "stress_overlap_sec": stress_overlap,
            "rest_overlap_sec": rest_overlap,
            "stress_ratio": stress_ratio,
            "rest_ratio": rest_ratio,
            "covered_ratio": covered_ratio,
            "dominant_phase": max(phase_overlaps, key=phase_overlaps.get) if phase_overlaps else "none",
            "phase_overlaps": str(phase_overlaps),
        })

        t0 += step_delta

    return pd.DataFrame(rows_no_discard), pd.DataFrame(rows_with_discard)


all_no_discard = []
all_with_discard = []
subject_summary_rows = []
skipped = []

subjects = sorted([
    d for d in os.listdir(STRESS_ROOT)
    if re.fullmatch(r"f\d{2}", d)
    and os.path.isdir(os.path.join(STRESS_ROOT, d))
])

print("v2 subjects:", subjects)
print("number of v2 subjects:", len(subjects))

for subject in subjects:
    subj_dir = os.path.join(STRESS_ROOT, subject)
    bvp_path = os.path.join(subj_dir, "BVP.csv")
    tags_path = os.path.join(subj_dir, "tags.csv")

    if not os.path.exists(bvp_path) or not os.path.exists(tags_path):
        skipped.append((subject, "missing BVP.csv or tags.csv"))
        continue

    try:
        bvp_start, bvp_end, fs, n_samples, bvp_duration = read_bvp_meta(bvp_path)
        tags = read_tags(tags_path)

        if len(tags) < 9:
            skipped.append((subject, f"tags count < 9 ({len(tags)})"))
            continue

        intervals_df = build_v2_intervals(tags)
        no_discard_df, with_discard_df = make_timeline_windows(
            intervals_df,
            window_sec=WINDOW_SEC,
            step_sec=STEP_SEC
        )

        no_discard_df.insert(0, "subject", subject)
        with_discard_df.insert(0, "subject", subject)

        all_no_discard.append(no_discard_df)
        all_with_discard.append(with_discard_df)

        nd_counts = no_discard_df["status_name"].value_counts().to_dict()
        wd_counts = with_discard_df["status_name"].value_counts().to_dict()
        wd_keep = with_discard_df[with_discard_df["keep"] == 1]["status_name"].value_counts().to_dict()

        subject_summary_rows.append({
            "subject": subject,
            "fs": fs,
            "n_samples": n_samples,
            "bvp_duration_min": bvp_duration / 60,
            "num_tags": len(tags),

            "no_discard_total": len(no_discard_df),
            "no_discard_normal": nd_counts.get("normal", 0),
            "no_discard_stress": nd_counts.get("stress", 0),

            "with_discard_total": len(with_discard_df),
            "with_discard_keep": int(with_discard_df["keep"].sum()),
            "with_discard_discard": wd_counts.get("discard", 0),
            "with_discard_normal": wd_keep.get("normal", 0),
            "with_discard_stress": wd_keep.get("stress", 0),
        })

        print("\n" + "=" * 80)
        print(f"[{subject}]")
        print(f"BVP duration : {bvp_duration/60:.2f} min ({sec_to_minsec(bvp_duration)})")
        print(f"timeline     : {intervals_df['start'].min()} ~ {intervals_df['end'].max()}")
        print(f"window/step  : {WINDOW_SEC}s / {STEP_SEC}s")

        print("\nNo-discard counts:")
        print(no_discard_df["status_name"].value_counts())

        print("\nWith-discard counts:")
        print(with_discard_df["status_name"].value_counts())

        print("\nWith-discard kept counts:")
        print(with_discard_df[with_discard_df["keep"] == 1]["status_name"].value_counts())

        print("\nStress windows preview with discard:")
        print(
            with_discard_df[with_discard_df["status_name"] == "stress"]
            [["window_index", "window_start", "window_end", "stress_ratio", "rest_ratio", "dominant_phase"]]
            .head(10)
            .to_string(index=False)
        )

    except Exception as e:
        skipped.append((subject, str(e)))


df_no_discard = pd.concat(all_no_discard, axis=0, ignore_index=True)
df_with_discard = pd.concat(all_with_discard, axis=0, ignore_index=True)
summary_df = pd.DataFrame(subject_summary_rows)

print("\n\n" + "#" * 80)
print("OVERALL SUMMARY")
print("#" * 80)

print("\nNo-discard overall:")
print(df_no_discard["status_name"].value_counts())

print("\nWith-discard overall:")
print(df_with_discard["status_name"].value_counts())

print("\nWith-discard kept overall:")
print(df_with_discard[df_with_discard["keep"] == 1]["status_name"].value_counts())

print("\nSubject summary:")
print(summary_df.to_string(index=False))

print("\nSkipped:")
for s, reason in skipped:
    print(f"{s}: {reason}")

df_no_discard.to_csv("v2_60s_30s_windows_no_discard.csv", index=False, encoding="utf-8-sig")
df_with_discard.to_csv("v2_60s_30s_windows_with_discard.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv("v2_60s_30s_window_label_summary.csv", index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" - v2_60s_30s_windows_no_discard.csv")
print(" - v2_60s_30s_windows_with_discard.csv")
print(" - v2_60s_30s_window_label_summary.csv")

v2 subjects: ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
number of v2 subjects: 17

[f01]
BVP duration : 54.21 min (54:12)
timeline     : 2013-06-12 16:25:46 ~ 2013-06-12 17:08:57
window/step  : 60s / 30s

No-discard counts:
status_name
normal    67
stress    18
Name: count, dtype: int64

With-discard counts:
status_name
normal     64
stress     18
discard     3
Name: count, dtype: int64

With-discard kept counts:
status_name
normal    64
stress    18
Name: count, dtype: int64

Stress windows preview with discard:
 window_index        window_start          window_end  stress_ratio  rest_ratio dominant_phase
            1 2013-06-12 16:25:46 2013-06-12 16:26:46           1.0         0.0           TMCT
            2 2013-06-12 16:26:16 2013-06-12 16:27:16           1.0         0.0           TMCT
            3 2013-06-12 16:26:46 2013-06-12 16:27:46           1.0         0.0           TMCT
            4 2013-06-12

### TPV Extraction

In [ ]:
!pip install ripser

In [3]:
import os
import re
import glob
import numpy as np
import pandas as pd

from pandas.errors import EmptyDataError
from ripser import ripser
from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist
from scipy.interpolate import interp1d


# =============================================================================
# CONFIG
# =============================================================================
DATA_ROOT = r"/content/drive/MyDrive/Colab Notebooks/datasets/Wearable-device-dataset/Wearable_Dataset"
STRESS_ROOT = os.path.join(DATA_ROOT, "STRESS")

OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/TPV"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# v2 only: f01~f18, but f14_a/f14_b excluded automatically
EXCLUDE_SUBJECTS = set()

FS_BVP = 64
FS_ACC = 32

WINDOW_SEC = 60
STEP_SEC = 30
STRESS_RATIO_THR = 0.50
REST_RATIO_THR = 0.10
USE_DISCARD = True   # True: ambiguous window 제거 / False: 무조건 다 라벨링

# Baseline excluded.
# With 60-sec windows, short stress tasks are mostly excluded naturally.
USE_SEGMENTS = {
    "First Rest",
    "Second Rest",
    "TMCT",
    "Real Opinion",
    "Opposite Opinion",
    "Subtract Test",
}

# ---- BVP filter ----
LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

# ---- Hard reject thresholds ----
FLATLINE_EPS = 1e-3
FLATLINE_MIN_SEC = 1.0
CLIPPING_RATIO_MAX = 0.10
MIN_PEAKS_PER_MIN = 30

# ---- Relaxed QC thresholds ----
VBR_MIN = 0.70
TC_MIN = 0.75
IBI_PASS_MIN = 0.80
MOTION_PERCENTILE = 80

# ---- Peak / beat extraction ----
MIN_HR_BPM = 40
MAX_HR_BPM = 180
MIN_PEAK_DISTANCE_SEC = 60.0 / MAX_HR_BPM
IBI_MIN_SEC = 0.30
IBI_MAX_SEC = 1.50

BEAT_BEFORE_SEC = 0.25
BEAT_AFTER_SEC = 0.45
BEAT_RESAMPLED_LEN = 100

# ---- Motion artifact thresholds ----
ALLOW_MISSING = False
ACC_DIFF_THRESH = 0.05
PSD_CORR_THRESH = 0.90
PSD_FMIN = 0.5
PSD_FMAX = 8.0

TPV_FEATURE_NAMES = [
    "birth_mean", "birth_std",
    "death_mean", "death_std",
    "lifetime_mean", "lifetime_std",
    "lifetime_max", "lifetime_min", "lifetime_median", "lifetime_iqr",
    "lifetime_skew", "lifetime_kurtosis",
    "birth_max", "birth_min", "birth_median", "birth_skew", "birth_kurtosis",
    "death_max", "death_min", "death_median", "death_skew", "death_kurtosis",
    "num_h1_features",
    "top1_lifetime", "top2_lifetime",
    "lifetime_max_div_min",
    "sum_squared_lifetimes",
    "persistent_entropy",
    "birth_value_entropy",
    "normalized_persistence_energy",
    "mean_pairwise_lifetime_distance",
    "gini_index",
    "lifetime_variance",
]

N_FEATURES = len(TPV_FEATURE_NAMES)
SELECTED_TPV_INDICES = list(range(N_FEATURES))



# =============================================================================
# Loaders for Empatica E4 files
# =============================================================================
def load_empatica_1d_csv(file_path, signal_name):
    raw = pd.read_csv(file_path, header=None)

    start_time = pd.to_datetime(raw.iloc[0, 0])
    fs = float(raw.iloc[1, 0])

    values = raw.iloc[2:, 0].astype(float).reset_index(drop=True).values
    timestamps = start_time + pd.to_timedelta(np.arange(len(values)) / fs, unit="s")

    df = pd.DataFrame({
        "time": timestamps,
        signal_name: values
    })

    return df, start_time, fs


def load_acc_csv(file_path):
    raw = pd.read_csv(file_path, header=None)

    start_time = pd.to_datetime(raw.iloc[0, 0])
    fs = float(raw.iloc[1, 0])

    data = raw.iloc[2:, :3].astype(float).reset_index(drop=True)
    timestamps = start_time + pd.to_timedelta(np.arange(len(data)) / fs, unit="s")

    df = pd.DataFrame({
        "time": timestamps,
        "acc_x": data.iloc[:, 0].values,
        "acc_y": data.iloc[:, 1].values,
        "acc_z": data.iloc[:, 2].values,
    })

    return df, start_time, fs


def load_tags_csv(file_path):
    if os.path.getsize(file_path) == 0:
        return []

    try:
        tags = pd.read_csv(file_path, header=None)
    except EmptyDataError:
        return []

    if tags.empty:
        return []

    tags[0] = pd.to_datetime(tags[0])
    return tags[0].tolist()


# =============================================================================
# v2 protocol intervals
# =============================================================================
def build_v2_stress_intervals(tags):
    """
    v2 mapping from checked protocol:
    tag index is 0-based.

    Exclude Baseline.

    TMCT             : tag[1] -> tag[2]
    First Rest       : tag[2] -> tag[3]
    Real Opinion     : tag[3] -> tag[4]
    Opposite Opinion : tag[5] -> tag[6]
    Second Rest      : tag[6] -> tag[7]
    Subtract Test    : tag[7] -> tag[8]
    """

    if len(tags) < 9:
        raise ValueError(f"v2 requires at least 9 tags, got {len(tags)}")

    specs = [
        ("TMCT",             tags[1], tags[2], 1),
        ("First Rest",       tags[2], tags[3], 0),
        ("Real Opinion",     tags[3], tags[4], 1),
        ("Opposite Opinion", tags[5], tags[6], 1),
        ("Second Rest",      tags[6], tags[7], 0),
        ("Subtract Test",    tags[7], tags[8], 1),
    ]

    rows = []
    for idx, (phase, start, end, label) in enumerate(specs):
        if phase not in USE_SEGMENTS:
            continue
        if end <= start:
            continue

        rows.append({
            "interval_idx": idx,
            "phase": phase,
            "start": start,
            "end": end,
            "label": int(label),
            "status_name": "stress" if label == 1 else "normal",
            "duration_sec": (end - start).total_seconds()
        })

    return pd.DataFrame(rows)


def overlap_sec(a_start, a_end, b_start, b_end):
    start = max(a_start, b_start)
    end = min(a_end, b_end)
    sec = (end - start).total_seconds()
    return max(0.0, sec)


def build_window_table_from_intervals(intervals_df, window_sec=60, step_sec=30):
    rows = []

    timeline_start = intervals_df["start"].min()
    timeline_end = intervals_df["end"].max()

    win_delta = pd.to_timedelta(window_sec, unit="s")
    step_delta = pd.to_timedelta(step_sec, unit="s")

    t0 = timeline_start
    window_idx = 0

    while t0 + win_delta <= timeline_end:
        t1 = t0 + win_delta
        window_idx += 1

        stress_overlap = 0.0
        rest_overlap = 0.0
        phase_overlaps = {}

        for _, r in intervals_df.iterrows():
            ov = overlap_sec(t0, t1, r["start"], r["end"])

            if ov <= 0:
                continue

            phase_overlaps[r["phase"]] = ov

            if int(r["label"]) == 1:
                stress_overlap += ov
            else:
                rest_overlap += ov

        stress_ratio = stress_overlap / window_sec
        rest_ratio = rest_overlap / window_sec
        covered_ratio = (stress_overlap + rest_overlap) / window_sec

        if USE_DISCARD:
            if stress_ratio >= STRESS_RATIO_THR:
                label = 1
                status_name = "stress"
            elif stress_ratio <= REST_RATIO_THR and rest_ratio > 0:
                label = 0
                status_name = "normal"
            else:
                t0 += step_delta
                continue
        else:
            if stress_overlap >= rest_overlap:
                label = 1
                status_name = "stress"
            else:
                label = 0
                status_name = "normal"

        dominant_phase = max(phase_overlaps, key=phase_overlaps.get) if phase_overlaps else "none"

        rows.append({
            "phase": dominant_phase,
            "label": label,
            "status_name": status_name,
            "window_start": t0,
            "window_end": t1,
            "major_ratio": max(stress_ratio, rest_ratio),
            "stress_ratio": stress_ratio,
            "rest_ratio": rest_ratio,
            "covered_ratio": covered_ratio,
        })

        t0 += step_delta

    return pd.DataFrame(rows)


# =============================================================================
# Basic utils
# =============================================================================
def fill_nan_with_median(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def fill_nan_acc_with_median(acc_xyz):
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32).copy()
    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return acc_xyz

    for c in range(3):
        ch = acc_xyz[:, c]
        med = np.nanmedian(ch)
        if np.isnan(med):
            med = 0.0
        acc_xyz[:, c] = np.nan_to_num(ch, nan=med, posinf=med, neginf=med)

    return acc_xyz


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


def zscore_1d(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    mu = float(np.mean(x))
    sd = float(np.std(x))
    if sd < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mu) / (sd + 1e-8)).astype(np.float32)


def safe_skew(x):
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x):
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def clamp01(x):
    return float(np.clip(x, 0.0, 1.0))


def compute_missing_ratio(x):
    x = np.asarray(x)
    return float(np.mean(~np.isfinite(x)))


# =============================================================================
# TPV extraction
# =============================================================================
def extract_tpv_features(sig_1d):
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)

    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    X = np.stack([sig[:-1], sig[1:]], axis=1)

    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))

    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (
            (2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted))
            / (n * np.sum(lifetimes_sorted))
            - (n + 1) / n
        )
        gini_index = float(gini)
    else:
        gini_index = 0.0

    sum_squared_lifetimes = float(np.sum(lifetimes ** 2))

    persistent_entropy = ph_entropy
    birth_value_entropy = betti_entropy

    normalized_persistence_energy = float(
        np.sum(lifetimes ** 2) / ((np.sum(lifetimes) ** 2) + 1e-8)
    )

    mean_pairwise_lifetime_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    lifetime_variance = float(np.var(lifetimes))

    lifetime_min = float(np.min(lifetimes))
    lifetime_max = float(np.max(lifetimes))
    lifetime_max_div_min = (
        float(lifetime_max / lifetime_min) if lifetime_min > 0 else 0.0
    )

    feats = [
        float(np.mean(births)),                         # 0
        float(np.std(births)),                          # 1
        float(np.mean(deaths)),                         # 2
        float(np.std(deaths)),                          # 3
        float(np.mean(lifetimes)),                      # 4
        float(np.std(lifetimes)),                       # 5
        lifetime_max,                                   # 6
        lifetime_min,                                   # 7
        float(np.median(lifetimes)),                    # 8
        float(iqr(lifetimes)),                          # 9
        safe_skew(lifetimes),                           # 10
        safe_kurtosis(lifetimes),                       # 11
        float(np.max(births)),                          # 12
        float(np.min(births)),                          # 13
        float(np.median(births)),                       # 14
        safe_skew(births),                              # 15
        safe_kurtosis(births),                          # 16
        float(np.max(deaths)),                          # 17
        float(np.min(deaths)),                          # 18
        float(np.median(deaths)),                       # 19
        safe_skew(deaths),                              # 20
        safe_kurtosis(deaths),                          # 21
        float(len(lifetimes)),                          # 22
        float(lifetimes_sorted[-1]),                    # 23
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,  # 24
        lifetime_max_div_min,                           # 25
        sum_squared_lifetimes,                          # 26
        persistent_entropy,                             # 27
        birth_value_entropy,                            # 28
        normalized_persistence_energy,                  # 29
        mean_pairwise_lifetime_distance,                # 30
        gini_index,                                     # 31
        lifetime_variance,                              # 32
    ]

    return np.asarray(feats, dtype=np.float32)


# =============================================================================
# QC helpers
# =============================================================================
def is_flatline(sig, fs, eps=FLATLINE_EPS, min_sec=FLATLINE_MIN_SEC):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    run_len = int(round(fs * min_sec))

    if len(sig) < run_len:
        return False

    ds = np.abs(np.diff(sig))
    low_change = ds < eps

    cnt = 0
    for v in low_change:
        if v:
            cnt += 1
            if cnt >= run_len - 1:
                return True
        else:
            cnt = 0

    return False


def clipping_ratio(sig):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)

    if len(sig) == 0:
        return 1.0

    lo = np.percentile(sig, 1)
    hi = np.percentile(sig, 99)

    if hi - lo < 1e-8:
        return 1.0

    near_lo = sig <= (lo + 0.01 * (hi - lo))
    near_hi = sig >= (hi - 0.01 * (hi - lo))

    return float(np.mean(near_lo | near_hi))


def detect_ppg_peaks(sig, fs):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)

    distance = max(1, int(round(MIN_PEAK_DISTANCE_SEC * fs)))
    prom = max(0.10, 0.20 * float(np.std(sig)))

    peaks, props = find_peaks(sig, distance=distance, prominence=prom)
    return peaks, props


def compute_motion_score(acc_xyz):
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)

    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return 1.0

    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))

    return float(np.median(dmag) + 0.5 * np.std(dmag))


def compute_acc_diff_metrics(acc_xyz):
    acc_xyz = np.asarray(acc_xyz, dtype=np.float32)

    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3 or len(acc_xyz) < 2:
        return {
            "acc_diff_mean": np.nan,
            "acc_diff_exceed_ratio": np.nan,
            "acc_diff_flag": 1
        }

    mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    dmag = np.abs(np.diff(mag))

    acc_diff_mean = float(np.mean(dmag)) if len(dmag) > 0 else 0.0
    acc_diff_exceed_ratio = float(np.mean(dmag > ACC_DIFF_THRESH)) if len(dmag) > 0 else 0.0
    acc_diff_flag = int(acc_diff_mean > ACC_DIFF_THRESH)

    return {
        "acc_diff_mean": acc_diff_mean,
        "acc_diff_exceed_ratio": acc_diff_exceed_ratio,
        "acc_diff_flag": acc_diff_flag
    }


def _safe_psd_1d(x, fs, fmin, fmax):
    x = np.asarray(x, dtype=np.float32).reshape(-1)

    if len(x) < 8:
        return None, None

    x = x[np.isfinite(x)]

    if len(x) < 8 or np.std(x) < 1e-8:
        return None, None

    nperseg = min(256, len(x))
    f, pxx = welch(x, fs=fs, nperseg=nperseg)

    mask = (f >= fmin) & (f <= min(fmax, fs / 2 - 1e-6))

    if np.sum(mask) < 4:
        return None, None

    return f[mask], pxx[mask]


def _corr_on_common_freq(f_ref, p_ref, f_other, p_other):
    if f_ref is None or f_other is None:
        return np.nan

    p_other_interp = np.interp(f_ref, f_other, p_other)

    a = np.log1p(p_ref)
    b = np.log1p(p_other_interp)

    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan

    r = np.corrcoef(a, b)[0, 1]
    return float(r) if np.isfinite(r) else np.nan


def compute_bvp_acc_psd_corr(seg_bvp_bp, seg_acc_xyz, fs_bvp, fs_acc, fmin=PSD_FMIN, fmax=PSD_FMAX):
    out = {
        "psd_corr_x": np.nan,
        "psd_corr_y": np.nan,
        "psd_corr_z": np.nan,
        "psd_corr_mag": np.nan,
        "psd_corr_max": np.nan,
        "psd_corr_flag": 1
    }

    seg_bvp_bp = np.asarray(seg_bvp_bp, dtype=np.float32).reshape(-1)
    acc_xyz = np.asarray(seg_acc_xyz, dtype=np.float32)

    if acc_xyz.ndim != 2 or acc_xyz.shape[1] != 3:
        return out

    f_bvp, p_bvp = _safe_psd_1d(seg_bvp_bp, fs_bvp, fmin, fmax)

    if f_bvp is None:
        return out

    acc_mag = np.sqrt(np.sum(acc_xyz ** 2, axis=1))
    corr_list = []

    for name, sig in [
        ("x", acc_xyz[:, 0]),
        ("y", acc_xyz[:, 1]),
        ("z", acc_xyz[:, 2]),
        ("mag", acc_mag),
    ]:
        f_acc, p_acc = _safe_psd_1d(sig, fs_acc, fmin, fmax)
        r = _corr_on_common_freq(f_bvp, p_bvp, f_acc, p_acc)
        out[f"psd_corr_{name}"] = r

        if np.isfinite(r):
            corr_list.append(r)

    if len(corr_list) == 0:
        return out

    out["psd_corr_max"] = float(np.max(corr_list))
    out["psd_corr_flag"] = int(out["psd_corr_max"] >= PSD_CORR_THRESH)

    return out


def extract_beats(sig, peaks, fs):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)

    beats = []
    before = int(round(BEAT_BEFORE_SEC * fs))
    after = int(round(BEAT_AFTER_SEC * fs))
    target_x = np.linspace(0, 1, BEAT_RESAMPLED_LEN)

    for p in peaks:
        s = p - before
        e = p + after

        if s < 0 or e > len(sig):
            continue

        beat = sig[s:e].copy()

        if len(beat) < 5:
            continue

        src_x = np.linspace(0, 1, len(beat))
        f = interp1d(src_x, beat, kind="linear")
        beats.append(f(target_x).astype(np.float32))

    if len(beats) == 0:
        return np.empty((0, BEAT_RESAMPLED_LEN), dtype=np.float32)

    return np.stack(beats, axis=0)


def median_template_corr(beats):
    if beats.ndim != 2 or len(beats) < 2:
        return 0.0

    template = np.median(beats, axis=0)
    vals = []

    for b in beats:
        if np.std(b) < 1e-8 or np.std(template) < 1e-8:
            vals.append(0.0)
        else:
            r = np.corrcoef(b, template)[0, 1]
            vals.append(float(r) if np.isfinite(r) else 0.0)

    return float(np.median(vals)) if len(vals) > 0 else 0.0


def compute_ibi_metrics(peaks, fs):
    peaks = np.asarray(peaks, dtype=np.int64)

    if len(peaks) < 3:
        return {
            "n_peaks": int(len(peaks)),
            "ibi_pass_ratio": 0.0,
            "valid_beat_ratio": 0.0,
            "ibi_plausibility": 0.0,
            "ibi_mean_sec": np.nan,
        }

    ibi = np.diff(peaks) / float(fs)
    plausible = (ibi >= IBI_MIN_SEC) & (ibi <= IBI_MAX_SEC)
    ibi_pass_ratio = float(np.mean(plausible)) if len(ibi) > 0 else 0.0

    if len(ibi) >= 3:
        med_ibi = np.median(ibi)
        valid = plausible & (np.abs(ibi - med_ibi) <= 0.30 * (med_ibi + 1e-8))
        vbr = float(np.mean(valid))
    else:
        vbr = ibi_pass_ratio

    return {
        "n_peaks": int(len(peaks)),
        "ibi_pass_ratio": ibi_pass_ratio,
        "valid_beat_ratio": vbr,
        "ibi_plausibility": ibi_pass_ratio,
        "ibi_mean_sec": float(np.mean(ibi)) if len(ibi) > 0 else np.nan,
    }


def notch_foot_peak_detectability_score(beats):
    if beats.ndim != 2 or len(beats) == 0:
        return 0.0

    scores = []

    for b in beats:
        foot = np.min(b[:max(5, len(b) // 4)])
        peak = np.max(b)
        amp = peak - foot
        dyn = np.max(b) - np.min(b) + 1e-8
        scores.append(clamp01(amp / dyn))

    return float(np.median(scores)) if len(scores) > 0 else 0.0


def compute_sqi_scores(vbr, tc, ibi_plausibility, motion_score, morph_score):
    motion_penalty = clamp01(motion_score)

    sqi_stress_basic = (
        0.40 * clamp01(vbr)
        + 0.30 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        - 0.10 * motion_penalty
    )

    sqi_stress_morph = (
        0.30 * clamp01(vbr)
        + 0.25 * clamp01(tc)
        + 0.20 * clamp01(ibi_plausibility)
        + 0.25 * clamp01(morph_score)
        - motion_penalty
    )

    return float(sqi_stress_basic), float(sqi_stress_morph), motion_penalty


def evaluate_window_quality(seg_bvp_raw, seg_acc_xyz, fs_bvp, fs_acc):
    out = {}

    out["bvp_missing_ratio"] = compute_missing_ratio(seg_bvp_raw)
    out["acc_missing_ratio"] = compute_missing_ratio(seg_acc_xyz)

    out["missing_flag"] = int(
        (out["bvp_missing_ratio"] > 0.0)
        or (out["acc_missing_ratio"] > 0.0)
    ) if not ALLOW_MISSING else 0

    seg_bvp = fill_nan_with_median(seg_bvp_raw)
    seg_acc_xyz = fill_nan_acc_with_median(seg_acc_xyz)

    seg_bvp_bp = bandpass_filter(
        seg_bvp,
        fs=fs_bvp,
        low=LOWCUT,
        high=HIGHCUT,
        order=FILTER_ORDER
    )

    seg_bvp_z = zscore_1d(seg_bvp_bp)

    out["flatline_flag"] = int(is_flatline(seg_bvp_bp, fs=fs_bvp))
    out["clipping_ratio"] = clipping_ratio(seg_bvp_bp)
    out["clipping_flag"] = int(out["clipping_ratio"] > CLIPPING_RATIO_MAX)

    peaks, _ = detect_ppg_peaks(seg_bvp_z, fs=fs_bvp)
    out["n_peaks"] = int(len(peaks))
    out["peak_sparse_flag"] = int(len(peaks) < MIN_PEAKS_PER_MIN)

    ibi_info = compute_ibi_metrics(peaks, fs_bvp)
    out.update(ibi_info)

    beats = extract_beats(seg_bvp_z, peaks, fs_bvp)

    tc = median_template_corr(beats)
    morph_score = notch_foot_peak_detectability_score(beats)
    motion_score_raw = compute_motion_score(seg_acc_xyz)

    out["median_template_corr"] = float(tc)
    out["notch_foot_peak_detectability"] = float(morph_score)
    out["motion_score_raw"] = float(motion_score_raw)

    sqi_basic, sqi_morph, motion_penalty = compute_sqi_scores(
        vbr=out["valid_beat_ratio"],
        tc=tc,
        ibi_plausibility=out["ibi_plausibility"],
        motion_score=motion_score_raw,
        morph_score=morph_score
    )

    out["motion_score"] = float(motion_penalty)
    out["SQI_stress_basic"] = float(sqi_basic)
    out["SQI_stress_morph"] = float(sqi_morph)

    acc_diff_info = compute_acc_diff_metrics(seg_acc_xyz)
    out.update(acc_diff_info)

    psd_corr_info = compute_bvp_acc_psd_corr(seg_bvp_bp, seg_acc_xyz, fs_bvp, fs_acc)
    out.update(psd_corr_info)

    hard_reject = (
        (out["missing_flag"] == 1)
        or (out["flatline_flag"] == 1)
        or (out["clipping_flag"] == 1)
        or (out["peak_sparse_flag"] == 1)
        or (out["acc_diff_flag"] == 1)
        or (out["psd_corr_flag"] == 1)
    )

    out["hard_reject"] = int(hard_reject)

    return seg_bvp_z, out


# =============================================================================
# Main extraction for v2 STRESS
# =============================================================================
def extract_subject_bvp_tpv(subject_dir):
    subject_id = os.path.basename(subject_dir)

    if subject_id in EXCLUDE_SUBJECTS:
        return pd.DataFrame()

    bvp_path = os.path.join(subject_dir, "BVP.csv")
    acc_path = os.path.join(subject_dir, "ACC.csv")
    tags_path = os.path.join(subject_dir, "tags.csv")

    if not (os.path.exists(bvp_path) and os.path.exists(acc_path) and os.path.exists(tags_path)):
        print(f"[WARN] {subject_id}: missing BVP/ACC/tags")
        return pd.DataFrame()

    tags = load_tags_csv(tags_path)

    if len(tags) < 9:
        print(f"[WARN] {subject_id}: invalid tags count={len(tags)}")
        return pd.DataFrame()

    bvp_df, _, fs_bvp = load_empatica_1d_csv(bvp_path, "bvp")
    acc_df, _, fs_acc = load_acc_csv(acc_path)

    if int(round(fs_bvp)) != FS_BVP:
        print(f"[WARN] {subject_id}: BVP fs mismatch. Expected={FS_BVP}, got={fs_bvp}")

    if int(round(fs_acc)) != FS_ACC:
        print(f"[WARN] {subject_id}: ACC fs mismatch. Expected={FS_ACC}, got={fs_acc}")

    intervals_df = build_v2_stress_intervals(tags)

    window_df = build_window_table_from_intervals(
        intervals_df,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC
    )

    rows = []
    window_counter = 0

    for _, w in window_df.iterrows():
        t0 = w["window_start"]
        t1 = w["window_end"]

        seg_bvp = bvp_df[
            (bvp_df["time"] >= t0) &
            (bvp_df["time"] < t1)
        ]["bvp"].values

        seg_acc = acc_df[
            (acc_df["time"] >= t0) &
            (acc_df["time"] < t1)
        ][["acc_x", "acc_y", "acc_z"]].values

        expected_bvp_len = int(WINDOW_SEC * FS_BVP)
        expected_acc_len = int(WINDOW_SEC * FS_ACC)

        if len(seg_bvp) != expected_bvp_len:
            continue

        if len(seg_acc) != expected_acc_len:
            continue

        seg_bvp_for_tpv, qc_info = evaluate_window_quality(
            seg_bvp_raw=seg_bvp,
            seg_acc_xyz=seg_acc,
            fs_bvp=FS_BVP,
            fs_acc=FS_ACC
        )

        tpv_full = extract_tpv_features(seg_bvp_for_tpv)

        window_counter += 1

        row = {
            "subject": subject_id,
            "window": f"window{window_counter}",
            "window_index": window_counter,
            "status": int(w["label"]),
            "status_name": "stress" if int(w["label"]) == 1 else "normal",
            "phase": w["phase"],
            "t_start": t0,
            "t_end": t1,
            "major_ratio": float(w["major_ratio"]),
            "stress_ratio": float(w["stress_ratio"]),
            "rest_ratio": float(w["rest_ratio"]),
            "covered_ratio": float(w["covered_ratio"]),
        }

        row.update(qc_info)

        for idx in SELECTED_TPV_INDICES:
            row[f"TPV{idx}"] = float(tpv_full[idx])

        rows.append(row)

    return pd.DataFrame(rows)


def apply_relaxed_personalized_qc(df):
    df = df.copy()

    if len(df) == 0:
        return df

    motion_thr_map = (
        df.groupby("subject")["motion_score"]
        .quantile(MOTION_PERCENTILE / 100.0)
        .to_dict()
    )

    df["motion_threshold_subject_p80"] = df["subject"].map(motion_thr_map)

    df["qc_relaxed_pass"] = (
        (df["hard_reject"] == 0)
        & (df["valid_beat_ratio"] >= VBR_MIN)
        & (df["median_template_corr"] >= TC_MIN)
        & (df["ibi_pass_ratio"] >= IBI_PASS_MIN)
        & (df["motion_score"] <= df["motion_threshold_subject_p80"])
    ).astype(int)

    return df


def main():
    subject_dirs = sorted([
        os.path.join(STRESS_ROOT, d)
        for d in os.listdir(STRESS_ROOT)
        if re.fullmatch(r"f\d{2}", d)
        and os.path.isdir(os.path.join(STRESS_ROOT, d))
    ])

    print(f"[INFO] Found v2 subject folders: {len(subject_dirs)}")
    print([os.path.basename(d) for d in subject_dirs])

    all_dfs = []

    for subject_dir in subject_dirs:
        sid = os.path.basename(subject_dir)

        if sid in EXCLUDE_SUBJECTS:
            print(f"[INFO] Skip excluded subject: {sid}")
            continue

        print(f"[INFO] Processing {sid} ...")

        try:
            df_sub = extract_subject_bvp_tpv(subject_dir)
            print(f"[INFO] {sid}: extracted {len(df_sub)} windows")

            if len(df_sub) > 0:
                print(df_sub["status_name"].value_counts().to_dict())
                print(df_sub["phase"].value_counts().to_dict())
                all_dfs.append(df_sub)

        except Exception as e:
            print(f"[WARN] Failed on {sid}: {e}")

    if len(all_dfs) == 0:
        print("[WARN] No windows extracted.")
        return

    df_all = pd.concat(all_dfs, axis=0, ignore_index=True)
    df_all = apply_relaxed_personalized_qc(df_all)

    base_cols = [
    "subject", "window", "window_index",
    "status", "status_name", "phase",
    "t_start", "t_end",
    "major_ratio", "stress_ratio", "rest_ratio", "covered_ratio"
    ]

    qc_cols = [
        "bvp_missing_ratio", "acc_missing_ratio", "missing_flag",
        "flatline_flag", "clipping_ratio", "clipping_flag",
        "n_peaks", "peak_sparse_flag",
        "ibi_pass_ratio", "valid_beat_ratio", "ibi_plausibility", "ibi_mean_sec",
        "median_template_corr", "notch_foot_peak_detectability",
        "acc_diff_mean", "acc_diff_exceed_ratio", "acc_diff_flag",
        "psd_corr_x", "psd_corr_y", "psd_corr_z", "psd_corr_mag",
        "psd_corr_max", "psd_corr_flag",
        "motion_score_raw", "motion_score",
        "SQI_stress_basic", "SQI_stress_morph",
        "hard_reject", "motion_threshold_subject_p80", "qc_relaxed_pass"
    ]

    feat_cols = [f"TPV{idx}" for idx in SELECTED_TPV_INDICES]

    final_cols = base_cols + qc_cols + feat_cols
    df_all = df_all[final_cols].copy()

    csv_all = os.path.join(OUTPUT_DIR, "Wearable_v2_BVP_TPV_noQC.csv")
    df_all.to_csv(csv_all, index=False, encoding="utf-8-sig")

    df_qc = df_all[df_all["qc_relaxed_pass"] == 1].copy()
    csv_qc = os.path.join(OUTPUT_DIR, "Wearable_v2_BVP_TPV_withQC.csv")
    df_qc.to_csv(csv_qc, index=False, encoding="utf-8-sig")

    summary = (
        df_all.groupby(["subject", "status_name"])
        .agg(
            n_total=("window", "count"),
            n_qc_pass=("qc_relaxed_pass", "sum")
        )
        .reset_index()
    )

    summary["retention_ratio"] = summary["n_qc_pass"] / summary["n_total"]

    summary_csv = os.path.join(OUTPUT_DIR, "Wearable_v2_BVP_TPV_summary.csv")
    summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

    phase_summary = (
        df_all.groupby(["subject", "phase", "status_name"])
        .agg(
            n_total=("window", "count"),
            n_qc_pass=("qc_relaxed_pass", "sum")
        )
        .reset_index()
    )

    phase_summary_csv = os.path.join(OUTPUT_DIR, "Wearable_v2_BVP_TPV_phase_summary.csv")
    phase_summary.to_csv(phase_summary_csv, index=False, encoding="utf-8-sig")

    print("\n[INFO] Saved:")
    print(csv_all)
    print(csv_qc)
    print(summary_csv)
    print(phase_summary_csv)

    print("\n[INFO] Status counts (all)")
    print(df_all["status_name"].value_counts())

    print("\n[INFO] Status counts (QC)")
    print(df_qc["status_name"].value_counts())

    print("\n[INFO] Phase counts (all)")
    print(df_all["phase"].value_counts())

    print("\n[INFO] Phase counts (QC)")
    print(df_qc["phase"].value_counts())

    print("\n[INFO] QC retention summary")
    print(summary)


if __name__ == "__main__":
    main()

[INFO] Found v2 subject folders: 17
['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
[INFO] Processing f01 ...
[INFO] f01: extracted 82 windows
{'normal': 64, 'stress': 18}
{'Second Rest': 37, 'First Rest': 27, 'TMCT': 15, 'Real Opinion': 2, 'Opposite Opinion': 1}
[INFO] Processing f02 ...
[INFO] f02: extracted 52 windows
{'normal': 38, 'stress': 14}
{'First Rest': 25, 'Second Rest': 13, 'TMCT': 12, 'Real Opinion': 1, 'Opposite Opinion': 1}
[INFO] Processing f03 ...
[INFO] f03: extracted 74 windows
{'normal': 64, 'stress': 10}
{'First Rest': 35, 'Second Rest': 29, 'TMCT': 8, 'Real Opinion': 1, 'Opposite Opinion': 1}
[INFO] Processing f04 ...
[INFO] f04: extracted 64 windows
{'normal': 53, 'stress': 11}
{'First Rest': 27, 'Second Rest': 26, 'TMCT': 8, 'Real Opinion': 1, 'Opposite Opinion': 1, 'Subtract Test': 1}
[INFO] Processing f05 ...
[INFO] f05: extracted 65 windows
{'normal': 52, 'stress': 13}
{'First Rest': 27,